## RAG with langchain and FAISS

In [2]:
#import load libraries

import os
from dotenv import load_dotenv
from typing import List,Dict,Any,Optional
import warnings
warnings.filterwarnings('ignore')

In [3]:
#langchain core importss
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage, HumanMessage

In [4]:
#langchain specifc import

from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq

In [5]:
#lanchain community import
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader


In [6]:
##lanchain chain imports

from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.embeddings import HuggingFaceHubEmbeddings

In [7]:
load_dotenv()

True

# Data Ingestion

In [8]:
sample_documents = [

    Document(
        page_content="""
BMW (Bayerische Motoren Werke AG) is a German luxury automobile manufacturer founded in 1916.
The company is headquartered in Munich, Germany.
BMW produces luxury cars, motorcycles, and electric vehicles.
Popular models include the BMW 3 Series, 5 Series, X5, and i4.
""",
        metadata={
            "source": "bmw_company_profile",
            "page": 1,
            "topic": "BMW Company Overview"
        }
    ),

    Document(
        page_content="""
The BMW i4 is an all-electric luxury sedan.
It offers a driving range of more than 300 miles depending on the variant.
BMW's electric vehicle lineup also includes the iX and i7 models.
""",
        metadata={
            "source": "bmw_ev_guide",
            "page": 1,
            "topic": "BMW Electric Vehicles"
        }
    ),

    Document(
        page_content="""
Groq is an AI technology company focused on high-performance inference.
Its Language Processing Unit (LPU) is designed to provide extremely low-latency AI responses.
Groq is widely used with open-source LLMs such as Llama and Qwen.
""",
        metadata={
            "source": "groq_overview",
            "page": 1,
            "topic": "Groq AI Platform"
        }
    ),

    Document(
        page_content="""
LangChain is an open-source framework for developing applications powered by large language models.
It provides components for prompts, chains, memory, agents, retrievers, and vector stores.
LangChain simplifies the implementation of Retrieval-Augmented Generation (RAG).
""",
        metadata={
            "source": "langchain_documentation",
            "page": 1,
            "topic": "LangChain Fundamentals"
        }
    ),

    Document(
        page_content="""
ChromaDB is an open-source vector database designed for AI applications.
It stores embeddings and enables semantic search over large document collections.
Chroma integrates easily with LangChain and other AI frameworks.
""",
        metadata={
            "source": "chromadb_guide",
            "page": 1,
            "topic": "ChromaDB Vector Store"
        }
    ),

    Document(
        page_content="""
Retrieval-Augmented Generation (RAG) combines information retrieval with language generation.
Relevant documents are retrieved from a vector database and provided to the language model as context.
RAG improves factual accuracy and reduces hallucinations.
""",
        metadata={
            "source": "rag_concepts",
            "page": 1,
            "topic": "RAG Architecture"
        }
    )

]

print(sample_documents)

[Document(metadata={'source': 'bmw_company_profile', 'page': 1, 'topic': 'BMW Company Overview'}, page_content='\nBMW (Bayerische Motoren Werke AG) is a German luxury automobile manufacturer founded in 1916.\nThe company is headquartered in Munich, Germany.\nBMW produces luxury cars, motorcycles, and electric vehicles.\nPopular models include the BMW 3 Series, 5 Series, X5, and i4.\n'), Document(metadata={'source': 'bmw_ev_guide', 'page': 1, 'topic': 'BMW Electric Vehicles'}, page_content="\nThe BMW i4 is an all-electric luxury sedan.\nIt offers a driving range of more than 300 miles depending on the variant.\nBMW's electric vehicle lineup also includes the iX and i7 models.\n"), Document(metadata={'source': 'groq_overview', 'page': 1, 'topic': 'Groq AI Platform'}, page_content='\nGroq is an AI technology company focused on high-performance inference.\nIts Language Processing Unit (LPU) is designed to provide extremely low-latency AI responses.\nGroq is widely used with open-source LLM

## Text Splitting

In [9]:
text_splitting = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len,
    separators=[" "]
)

chunks = text_splitting.split_documents(sample_documents)
chunks

[Document(metadata={'source': 'bmw_company_profile', 'page': 1, 'topic': 'BMW Company Overview'}, page_content='BMW (Bayerische Motoren Werke AG) is a German luxury automobile manufacturer founded in 1916.\nThe company is headquartered in Munich, Germany.\nBMW produces luxury cars, motorcycles, and electric vehicles.\nPopular models include the BMW 3 Series, 5 Series, X5, and i4.'),
 Document(metadata={'source': 'bmw_ev_guide', 'page': 1, 'topic': 'BMW Electric Vehicles'}, page_content="The BMW i4 is an all-electric luxury sedan.\nIt offers a driving range of more than 300 miles depending on the variant.\nBMW's electric vehicle lineup also includes the iX and i7 models."),
 Document(metadata={'source': 'groq_overview', 'page': 1, 'topic': 'Groq AI Platform'}, page_content='Groq is an AI technology company focused on high-performance inference.\nIts Language Processing Unit (LPU) is designed to provide extremely low-latency AI responses.\nGroq is widely used with open-source LLMs such a

In [10]:
len(chunks)

6

## Load the embedding model

In [11]:
from langchain.embeddings import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embedding

/var/folders/1b/83xpzy2s1zz2hh1vm7mnkylm0000gn/T/ipykernel_1111/3447641268.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 27576.49it/s]


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [12]:
#example for checking embeding is working or not

text = "Hai how are you ?"
test_embedding = embedding.embed_query(text)
test_embedding

[-0.044084616005420685,
 0.0011714411666616797,
 0.06183938309550285,
 0.009126780554652214,
 -0.04676127806305885,
 0.007407549303025007,
 0.12631605565547943,
 -0.03460094705224037,
 -0.05975912883877754,
 0.0067335511557757854,
 0.04568314924836159,
 -0.08528073132038116,
 0.0038299509324133396,
 -0.010644541122019291,
 0.07788016647100449,
 -0.011862418614327908,
 0.04340963065624237,
 -0.13898125290870667,
 -0.13070249557495117,
 0.02587183192372322,
 -0.04331399127840996,
 0.0633673146367073,
 0.023770330473780632,
 0.05034218356013298,
 -0.04178664833307266,
 0.016437264159321785,
 -0.036325644701719284,
 -0.02572707086801529,
 0.001956644468009472,
 0.00905621238052845,
 -0.00877555925399065,
 0.10850900411605835,
 -0.06991955637931824,
 0.03499124199151993,
 -0.01853928714990616,
 -0.00747423293069005,
 -0.039544545114040375,
 -0.10016688704490662,
 -0.019884208217263222,
 -0.019248666241765022,
 -0.015643900260329247,
 -0.005455428268760443,
 0.02272934466600418,
 -0.09072738

## Create FAISS vector store

In [13]:
vectorestore = FAISS.from_documents(
    documents= chunks,
    embedding=embedding
)
print(f"Vector created count : {vectorestore.index.ntotal}")

Vector created count : 6


In [14]:
#save the vector file for the later use

vectorestore.save_local("faiss_index")
print("Vectore stored !!")

Vectore stored !!


In [15]:
#load the saved faiss

loaded_vectorstore = FAISS.load_local(
    "faiss_index",
    embedding,
    allow_dangerous_deserialization=True

)

print(f"The actual vectors stored from the local : {loaded_vectorstore.index.ntotal}")

The actual vectors stored from the local : 6


In [16]:
##similarity search

query = "what is chromaDB ?"
result = vectorestore.similarity_search(query,k=2)
result

[Document(id='ba49cdb8-a959-4b13-a4ee-42fbdd065a90', metadata={'source': 'chromadb_guide', 'page': 1, 'topic': 'ChromaDB Vector Store'}, page_content='ChromaDB is an open-source vector database designed for AI applications.\nIt stores embeddings and enables semantic search over large document collections.\nChroma integrates easily with LangChain and other AI frameworks.'),
 Document(id='0bb040e5-e0cc-4fa6-9601-a77f37d9d28a', metadata={'source': 'langchain_documentation', 'page': 1, 'topic': 'LangChain Fundamentals'}, page_content='LangChain is an open-source framework for developing applications powered by large language models.\nIt provides components for prompts, chains, memory, agents, retrievers, and vector stores.\nLangChain simplifies the implementation of Retrieval-Augmented Generation (RAG).')]

In [17]:
#similarity search with score

query1= "what is RAG ?"
result1 = vectorestore.similarity_search_with_score(query1,k=2)
result1

[(Document(id='fb94ef72-78de-4825-963d-1162b87c6680', metadata={'source': 'rag_concepts', 'page': 1, 'topic': 'RAG Architecture'}, page_content='Retrieval-Augmented Generation (RAG) combines information retrieval with language generation.\nRelevant documents are retrieved from a vector database and provided to the language model as context.\nRAG improves factual accuracy and reduces hallucinations.'),
  np.float32(1.1124604)),
 (Document(id='0bb040e5-e0cc-4fa6-9601-a77f37d9d28a', metadata={'source': 'langchain_documentation', 'page': 1, 'topic': 'LangChain Fundamentals'}, page_content='LangChain is an open-source framework for developing applications powered by large language models.\nIt provides components for prompts, chains, memory, agents, retrievers, and vector stores.\nLangChain simplifies the implementation of Retrieval-Augmented Generation (RAG).'),
  np.float32(1.3793314))]

In [18]:
##filter by the metadata

filt_dict = {"source":"rag_concepts"}
filtered = vectorestore.similarity_search(
    query1,
    k=2,
    filter=filt_dict
)
print(filtered)

[Document(id='fb94ef72-78de-4825-963d-1162b87c6680', metadata={'source': 'rag_concepts', 'page': 1, 'topic': 'RAG Architecture'}, page_content='Retrieval-Augmented Generation (RAG) combines information retrieval with language generation.\nRelevant documents are retrieved from a vector database and provided to the language model as context.\nRAG improves factual accuracy and reduces hallucinations.')]


# Build RAG Chain LCEL

In [19]:
#load the llm

llm = ChatGroq(
    model_name="qwen/qwen3-32b"
)
llm.invoke("hi")

AIMessage(content='<think>\nOkay, the user said "hi". That\'s a greeting. I should respond in a friendly and welcoming way. Let me make sure to acknowledge their greeting and offer assistance. Maybe something like "Hello! How can I assist you today?" That\'s open-ended and invites them to ask questions or share what they need help with. I should keep it simple and positive. No need for any extra information unless they ask. Just a standard friendly response.\n</think>\n\nHello! How can I assist you today? 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 9, 'total_tokens': 115, 'completion_time': 0.209066581, 'prompt_time': 0.000476493, 'queue_time': 0.488055147, 'total_time': 0.209543074}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019e8e0d-f263-7c30-9029-0739a606ea5d-0', usage_metadata={'input_tokens': 9, 'output_tokens': 

## Simple RAG chain

In [20]:
simple_rag_propmt   = ChatPromptTemplate.from_template(
    """Answer the question based on the context
    Context: {context}
    Question: {question}
    Answer:
    """
)

In [21]:
retriever = vectorestore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [22]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x127d3cd50>, search_kwargs={'k': 3})

In [23]:
def formated_docs(docs: List[Document]) -> str:
    """formated documents for insertion into prompt"""
    formatted=[]
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'Unknown')
        formatted.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}")
    return "\n\n".join(formatted)

In [24]:
simple_rag_chain = (
    {
        "context": retriever | formated_docs ,
        "question":RunnablePassthrough()
    }
    | simple_rag_propmt
    | llm
    | StrOutputParser()
)

In [25]:
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x127d3cd50>, search_kwargs={'k': 3})
           | RunnableLambda(formated_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the context\n    Context: {context}\n    Question: {question}\n    Answer:\n    '), additional_kwargs={})])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x13a401190>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x13a49c350>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)
| StrOutputParser()

In [ ]:
converstional_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an AI assistant for question answering tasks. Use the retrieved context to answer the question."),
    ("placeholder", "{chat_history}"),
    ("human", "Context : {context}\n\n Question : {input}"),

])

In [42]:
def create_conversational_rag():
    return(
        RunnablePassthrough.assign(
            context=lambda x: formated_docs(retriever.invoke(x["input"]))
        )
        | converstional_prompt
        | llm
        | StrOutputParser()
    )
conversational_rag = create_conversational_rag()


In [30]:
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: formated_docs(retriever.invoke['input']))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='

In [31]:
#streaming RAG Chain

stream_rag = (
    {
        "context": retriever | formated_docs, "question": RunnablePassthrough()
    }
    | simple_rag_propmt
    | llm
)

In [38]:
q = "what is groq ?"
ans = simple_rag_chain.invoke(q)
print(ans)

print("====" *50)

for chunks in stream_rag.stream(q):
    print(chunks.content, end="", flush=True)

<think>
Okay, let's tackle this question. The user is asking what Groq is. I need to base my answer on the provided context.

Looking at Document 1, it says Groq is an AI tech company focused on high-performance inference. Their main product is the Language Processing Unit (LPU) designed for low-latency AI responses. Also mentioned that they work with open-source LLMs like Llama and Qwen. 

So, the key points from Document 1 are: AI company, high-performance inference, LPU with low latency, compatibility with open-source models. The other documents mention LangChain and RAG, but the question is specifically about Groq. Since the user is asking for a definition of Groq, I should focus only on Document 1. 

No need to include information from the other documents unless it's relevant. But since the question is just about Groq, the answer should come from Document 1. Let me make sure I don't add anything extra from the other sources. 

So, the answer should state Groq's focus area, their L

In [43]:
##convestional rag

chat_history=[]
q1 = "what is groq ?"
a1 = conversational_rag.invoke({
    "input": q1,
    "chat_history": chat_history
})

In [44]:
print(a1)

<think>
Okay, the user is asking "what is Groq?" Let me look at the provided documents to find the answer.

First, Document 1 from the groq_overview says that Groq is an AI technology company focused on high-performance inference. Their product is the Language Processing Unit (LPU), which is designed for low-latency AI responses. It also mentions that Groq is used with open-source LLMs like Llama and Qwen. That's the main info here.

Documents 2 and 3 talk about LangChain and RAG concepts, which don't directly relate to Groq. The question is specifically about Groq, so I can ignore those unless there's a connection. But the user didn't mention LangChain or RAG in their question, so probably not needed here.

I need to make sure the answer is based solely on Document 1. The key points are: AI tech company, high-performance inference, LPU with low latency, and compatibility with open-source LLMs. I should present this in a clear, concise way without adding info from the other documents. 

In [46]:
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=a1)
])

In [47]:
q2 = "who build this ?"
a2 = conversational_rag.invoke({
    "input": q2,
    "chat_history": chat_history
})

In [48]:
a2

'<think>\nOkay, the user asked, "who build this?" but it\'s a bit vague. They might be referring to the previous answer about Groq or the BMW information. Let me check the context again.\n\nLooking at the history, the user first asked about Groq, and I explained it\'s an AI company with LPUs. Then they provided three documents: BMW, Groq, and LangChain. The current question is "who build this?" without specifying. Since the last context included BMW, Groq, and LangChain, but the immediate previous answer was about Groq, maybe they\'re asking about Groq\'s creation.\n\nAlternatively, if they\'re referring to BMW, the answer would be different. But the user\'s question is short, so I need to clarify. However, the user might not want to be told to clarify. Maybe I should infer based on the documents provided. The documents mention BMW, Groq, and LangChain. The last answer was about Groq, so it\'s possible they\'re asking who built Groq.\n\nBut the user might be confused and not specify. S